In [10]:
### look at moisturizer's full ingredients list, creat dictionary, and see how many times those values appear in that dictionary
##look at interesting/unique ingredients in moisturizers such as alcohol and fragrance and see how many times those ingredients appear in the moisturizers data set

In [11]:
import pandas as pd
df = pd.read_csv('/Users/niyadoyle/adv_python/cuny-advanced-data-journalism-fall-2026/moisturizersassignment/product_info.csv')
print(df.shape)

(8494, 27)


In [12]:
df.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='str')

In [13]:
skincare_ingredients = df[df[ 'primary_category'] == 'Skincare']
print(skincare_ingredients)

     product_id                                       product_name  brand_id  \
89      P439055               GENIUS Sleeping Collagen Moisturizer      6018   
90      P421277                       GENIUS Liquid Collagen Serum      6018   
91      P467602            Triple Algae Eye Renewal Balm Eye Cream      6018   
92      P432045               GENIUS Liquid Collagen Lip Treatment      6018   
93      P311143  SUBLIME DEFENSE Ultra Lightweight UV Defense F...      6018   
...         ...                                                ...       ...   
8424    P501474  The Youth Vault: 13-Piece Vegan Skincare + App...      6169   
8425    P502998  Youth Stacks: Plump It Up for Dry, Dehydrated ...      6169   
8426    P503872     Mini Mandelic Acid + Superfood Unity Exfoliant      6169   
8427    P502977  Youth Stacks: Daily Skin Health Your Way for P...      6169   
8428    P502978   Youth Stacks: Brighter Tomorrow Duo for Dullness      6169   

               brand_name  loves_count 

In [14]:
skincare_products = df.loc[
    df['primary_category'].astype('string').str.strip().str.casefold().eq('skincare'),
    ['product_name', 'brand_name', 'ingredients', 'price_usd']
]

print(skincare_products)

                                           product_name           brand_name  \
89                 GENIUS Sleeping Collagen Moisturizer             Algenist   
90                         GENIUS Liquid Collagen Serum             Algenist   
91              Triple Algae Eye Renewal Balm Eye Cream             Algenist   
92                 GENIUS Liquid Collagen Lip Treatment             Algenist   
93    SUBLIME DEFENSE Ultra Lightweight UV Defense F...             Algenist   
...                                                 ...                  ...   
8424  The Youth Vault: 13-Piece Vegan Skincare + App...  Youth To The People   
8425  Youth Stacks: Plump It Up for Dry, Dehydrated ...  Youth To The People   
8426     Mini Mandelic Acid + Superfood Unity Exfoliant  Youth To The People   
8427  Youth Stacks: Daily Skin Health Your Way for P...  Youth To The People   
8428   Youth Stacks: Brighter Tomorrow Duo for Dullness  Youth To The People   

                                       

In [26]:
#filtering the data to only include moisturizers
moisturizers = df[df['secondary_category'] == 'Moisturizers']
print(len(moisturizers))

551


In [ ]:
import ast, re
from collections import defaultdict

def split_outside_parens(text):
    parts, depth, current = [], 0, []
    for ch in text:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth = max(depth - 1, 0)
        if ch == ',' and depth == 0:
            parts.append(''.join(current))
            current = []
        else:
            current.append(ch)
    parts.append(''.join(current))
    return parts

def clean(ing):
    ing = ing.strip().lower()
    ing = re.sub(r'\s*\(?\d+(\.\d+)?\s*%\)?', '', ing)   # drop percentages like 2% or (1.5%)
    ing = ing.rstrip('.*').strip()
    if re.fullmatch(r'(purified |distilled )?water\b.*|aqua\b.*', ing) and len(ing) < 30:
        return 'water'                                   # water/aqua/eau variants -> 'water'
    return ing

def parse_ingredients(raw):
    if not isinstance(raw, str):                         # handles NaN
        return []
    raw = re.sub(r'\\(?![\'"])', '/', raw)               # fix stray backslashes (removes the warning)
    try:
        items = ast.literal_eval(raw)                    # string -> real list
    except (ValueError, SyntaxError):
        items = [raw.strip("[]'\"")]
    cleaned = []
    for item in items:
        item = item.strip()
        if item.endswith(':') or len(item) > 400:        # skip kit labels / descriptive text
            continue
        for ing in split_outside_parens(item):
            ing = clean(ing)
            if ing and len(ing) < 80:                    # skip leftover sentences
                cleaned.append(ing)
    return cleaned

#filtering the data to only include moisturizers
moisturizers = df[df['secondary_category'] == 'Moisturizers'].copy()
print(len(moisturizers), "moisturizer products")

moisturizers['ingredient_list'] = moisturizers['ingredients'].apply(parse_ingredients)

# build a dictionary of moisturizer ingredients and # of moisturizers
moisturizer_ingredients = defaultdict(list)
for name, ings in zip(moisturizers['product_name'], moisturizers['ingredient_list']):
    for ing in set(ings):
        moisturizer_ingredients[ing].append(name)

print(len(moisturizer_ingredients), "unique ingredients")

551 moisturizer products
461 unique ingredients


In [30]:
###now time to put this dictionary into a dataframe and see how many times each ingredient appears in the moisturizers data set

ing_df = pd.DataFrame(
    [(ing, len(prods)) for ing, prods in skincare_ingredients.items()],
    columns=['ingredient', 'num_products']
).sort_values('num_products', ascending=False)

pd.set_option('display.max_rows', None)  
ing_df

,ingredient,num_products
62,water,1724
4,linalool,1047
18,limonene,991
9,citronellol,856
5,geraniol,845
1,citral,792
131,phenoxyethanol,600
21,coumarin,566
15,benzyl salicylate,560
66,alcohol denat,557
